In [2]:
"""
Module: Model Training & Export Pipeline for Teleprzyjaciele_IBM_Z_Datathon_2026_Project
Description: This script loads a downsized PaySim dataset, performs feature engineering,
and trains a lightweight Feed-Forward Neural Network (FFNN) for fraud detection.
Crucially, it exports the trained model to the ONNX format, which is required for
achieving real-time, ultra-low latency scoring (e.g., bringing latency down to 1ms)
when deployed on IBM Z / LinuxONE architectures leveraging the Telum processor
Theme: Real-Time AI for Critical Decisions
"""

import os
# ------Forced CPU execution--------
# IBM Z's Telum processor handles AI inferencing directly on-chip rather than relying on external GPUs.
# Disabling CUDA prevents environment initialization errors when porting our code to the s390x architecture.
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
import tf2onnx

DATA_PATH = '../data/paysim_downsized.csv'
MODEL_OUTPUT_PATH = '../models/fraud_detect.onnx'

def prepare_data(file_path: str) -> tuple[np.ndarray, np.ndarray]:
    """
    Loads raw transaction data and applies essential feature engineering.
    Optimized for processing financial data commonly found in systems where 90%
    of global credit card transactions occur (like IBM Z)
    """
    df = pd.read_csv(file_path)

    # Categorical mapping of transaction types to numeric values for the neural network.
    tx_mapping = {"TRANSFER": 1.0, "CASH_OUT": 2.0, "PAYMENT": 3.0, "DEBIT": 4.0, "CASH_IN": 5.0}
    df['type_enc'] = df['type'].map(tx_mapping).fillna(0.0)

    # Logarithmic transformation flattens massive variance in transaction amounts,
    # helping the model detect mathematical anomalies faster.
    df['log_amount'] = np.log1p(df['amount'])

    # drain_ratio: A powerful hackathon heuristic to catch Account Takeover (ATO) attacks.
    # Added a tiny epsilon (1e-9) to the denominator to mathematically prevent ZeroDivisionError crashes.
    df['drain_ratio'] = df['amount'] / (df['oldbalanceOrg'] + 1e-9)

    features = [
        'type_enc', 'log_amount', 'oldbalanceOrg', 'newbalanceOrig',
        'oldbalanceDest', 'newbalanceDest', 'drain_ratio'
    ]

    # Return as float32 tensors to ensure strict compatibility with ONNX and TensorFlow graph expectations.
    return df[features].values.astype(np.float32), df['isFraud'].values.astype(np.float32)

def build_model(input_dim: int) -> Sequential:
    """
    Constructs a lightweight Feed-Forward Neural Network architecture.
    Designed specifically for millisecond-level inference speeds on enterprise transactional systems.
    """
    model = Sequential([
        tf.keras.Input(shape=(input_dim,), name="input_features"),
        Dense(32, activation='relu'),
        # Dropout disables 20% of neurons randomly during training to force the network
        # to learn generalized fraud patterns rather than memorizing the training set (preventing overfitting).
        Dropout(0.2),
        Dense(16, activation='relu'),
        # Sigmoid activation cleanly compresses the final output into a 0.0 to 1.0 fraud probability score.
        Dense(1, activation='sigmoid', name="fraud_probability")
    ])

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

def export_to_onnx(keras_model: Sequential, output_path: str, input_dim: int) -> None:
    """
    Safely exports the model to the environment-agnostic ONNX format supported by IBM Z
    We use direct graph mapping (tf.function) to bypass known tf2onnx compatibility issues with Keras 3.
    """
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    #Encapsulation of the model inside a static TensorFlow
    @tf.function
    def model_fn(x):
        # CRITICAL FOR PRODUCTION: setting training=False.
        # traning = true is learning mode, dropout of 20% of neurons to force netowrk to learn robust patterns
        # tratning = false is locks the model into prediction mode and completely disables the droout of neurons. Model is deterministic now.
        return keras_model(x, training=False)

    # None allows for batch processing or single-transaction real-time scoring.
    spec = [tf.TensorSpec((None, input_dim), tf.float32, name="input_features")]

    # Convert directly from the static function graph to ONNX
    tf2onnx.convert.from_function(
        model_fn,
        input_signature=spec,
        output_path=output_path,
        opset=13
    )

if __name__ == "__main__":
    # Initializtion of the data pipeline
    X, y = prepare_data(DATA_PATH)

    # buildng and traning
    model = build_model(input_dim=X.shape[1])
    # validation_split=0.2 acts as a quick sanity check to monitor for overfitting.
    model.fit(X, y, epochs=5, batch_size=256, validation_split=0.2)

    # export to onnx
    export_to_onnx(model, MODEL_OUTPUT_PATH, input_dim=X.shape[1])

    print(f"SUCCESS: Production model successfully exported to {MODEL_OUTPUT_PATH}")

Epoch 1/5
238/238 ━━━━━━━━━━━━━━━━━━━━ 1s 936us/step - accuracy: 0.9505 - loss: 353351368704.0000 - val_accuracy: 0.9840 - val_loss: 17482813440.0000
Epoch 2/5
238/238 ━━━━━━━━━━━━━━━━━━━━ 0s 722us/step - accuracy: 0.9763 - loss: 19228301312.0000 - val_accuracy: 0.9849 - val_loss: 16095387648.0000
Epoch 3/5
238/238 ━━━━━━━━━━━━━━━━━━━━ 0s 724us/step - accuracy: 0.9568 - loss: 15249133568.0000 - val_accuracy: 0.9364 - val_loss: 13684029440.0000
Epoch 4/5
238/238 ━━━━━━━━━━━━━━━━━━━━ 0s 724us/step - accuracy: 0.9397 - loss: 15478689792.0000 - val_accuracy: 0.9199 - val_loss: 10932018176.0000
Epoch 5/5
238/238 ━━━━━━━━━━━━━━━━━━━━ 0s 719us/step - accuracy: 0.9330 - loss: 12052893696.0000 - val_accuracy: 0.9223 - val_loss: 11088261120.0000
SUCCESS: Production model successfully exported to ../models/fraud_detect.onnx


I0000 00:00:1792224098.509532   22821 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0
I0000 00:00:1792224098.509597   22821 single_machine.cc:376] Starting new session
I0000 00:00:1792224098.518278   22821 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0
I0000 00:00:1792224098.518328   22821 single_machine.cc:376] Starting new session
